# Calibration ablation across four backbones

The four-backbone grid answered the *representation* half: no robust training method reduces the
accuracy-matched divergence on CelebA, at any of the four backbones. It cannot answer the
*calibration* half, because it only ever evaluated `marginal_split`.

The calibration evidence currently rests on `calibration_ablation.csv`, which covers **two**
backbones and **CelebA only**. So the paper's central claim -- worst-group coverage is delivered
by the calibration mechanism, not by training -- has never been tested on DINOv2 or ViT-B/16, and
that is exactly what R1.3 asks about. This notebook closes that gap: 4 backbones x 2 datasets x
3 calibration policies.

**Two changes to the ablation code were needed first**, both verified before writing this:

| change | why |
|---|---|
| the gate now KEEPS the records of excluded arms, tagged | `c1_verdict` requires every method to be present, and ERM -- whose marginal-split failure *is* the claim -- is gated out on CelebA for DINOv2 and ViT. Without this the two new cells would silently lose their headline method. |
| per-cell streaming + resume | the previous version held every GridData at once and wrote nothing until the end; that is what exhausted RAM and lost a run before. |

`verify_ablation.py` checks streaming against the batched path: 16/16, records identical at full
float precision, gate semantics preserved, a partial-then-resumed run equal to one clean run, and
CSV round-trip drift 0.00e+00.

It also stores the diagnostic columns the old schema dropped -- `n_cal_worst_group` above all,
which is what the Mondrian small-group question needs and cannot be recovered afterwards.

## 0. Parameters -- **EDIT THESE**

In [ ]:
REPO_URL      = "https://github.com/octadion/vgscp"
REPO_BRANCH   = "main"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
WATERBIRDS_URL= "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE = "kaggle"       # the Drive-cached zip is used first; no credential needed
CELEBA_DRIVE  = ""

# --- the grid ----------------------------------------------------------------------
# Four backbones spanning two architecture families AND three pretraining regimes, which is what
# makes R1.3 ("only two backbones limits the generalizability") an answered point rather than a
# conceded one:
#   resnet50_erm   CNN  supervised, fine-tuned in-domain
#   clip_vitb32    ViT  image-text contrastive
#   dinov2_vitb14  ViT  self-supervised, never saw a label
#   vit_b16_in1k   ViT  supervised ImageNet
BACKBONES = ("resnet50_erm", "clip_vitb32", "dinov2_vitb14", "vit_b16_in1k")
DATASETS  = ("waterbirds", "celeba")
SEEDS     = (0, 1, 2, 3, 4)    # R1.4 asked for more than the submitted three
METHODS   = ("erm", "dfr", "afr", "groupdro_ll", "balanced_subsample")
SCORES    = ("APS", "RAPS", "THR")
RHO_SWEEP = (0.95, 0.9, 0.8, 0.7, 0.6, 0.5)
N_SPLITS  = 10
ALPHA     = 0.1

CELEBA_RESNET_MAX_TRAIN = 30000   # must MATCH the paper's original run, or the ResNet cache misses
                                  # and this CPU runtime would start training a ResNet-50.

# The 50k CelebA representation ablation (optional, free: pure cache hits). It recovers the records
# for the abandoned subsample protocol, under which three cells read "marginal WINS" -- a
# counterexample that disappears at full train. Worth keeping as evidence that the full-train
# decision was the right one; the 7.5 GB of features can then be deleted.
RUN_50K_ABLATION = True
CEL50_EPOCHS = {"erm": 5, "reweight": 5, "groupdro": 8}   # the old budget, to match the old keys
CEL50_SEEDS  = (0, 1, 2)

## 1. Drive, repo, caches

In [ ]:
import os, sys, time, subprocess

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-2000:])
    if r.returncode != 0:
        print(f"[shell FAILED rc={r.returncode}] {cmd}")
        if r.stderr.strip(): print(r.stderr[-2000:])
        if check: raise RuntimeError(f"command failed: {cmd}")
    return r.returncode == 0

def init_drive(mount="/content/drive", retries=3):
    from google.colab import drive
    for attempt in range(1, retries + 1):
        try:
            drive.mount(mount, force_remount=attempt > 1)
            os.makedirs(DRIVE_CACHE, exist_ok=True)
            probe, tok = os.path.join(DRIVE_CACHE, ".mount_probe"), str(time.time())
            with open(probe, "w") as fh: fh.write(tok)
            with open(probe) as fh: got = fh.read()
            os.remove(probe)
            if got != tok: raise IOError("probe read-back mismatch")
            print(f"Drive OK (attempt {attempt})")
            return
        except Exception as e:
            print(f"[drive] attempt {attempt}/{retries}: {e}"); time.sleep(5 * attempt)
    raise RuntimeError("Drive would not mount.")

init_drive()
REPO_DIR = "/content/vgscp"
sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

os.makedirs("results", exist_ok=True)
for c in ("cache_clip", "cache_resnet", "cache_frozen", "cache_finetune", "study"):
    tgt = f"{DRIVE_CACHE}/{c}"; os.makedirs(tgt, exist_ok=True)
    sh(f"rm -rf results/{c}"); sh(f"ln -s {tgt} results/{c}")
    probe = f"results/{c}/.link_probe"
    with open(probe, "w") as fh: fh.write("ok")
    assert os.path.exists(f"{tgt}/.link_probe"), f"results/{c} does not resolve to Drive"
    os.remove(probe)

try:
    import torch
    gpu = torch.cuda.is_available()
except Exception:
    gpu = False
print(f"\nrepo: {os.getcwd()}")
print(f"GPU present: {gpu}  (not needed -- every backbone should be a cache hit)")
print(f"vCPUs: {os.cpu_count()}")

# Measured per arm, not estimated -- two earlier estimates here were wrong. The worst CelebA cell
# is resnet50_erm (d=2048): GridData 1.55 GiB resident, plus the transient an arm needs to fit a
# head on the 162,770 x 2048 train split. That transient was 4.3x the input (a float64 upcast in
# the L2 guard, and numpy std's temporary); both are gone and it is now 2.4x, taking the cell peak
# from 7.3 to 5.0 GiB. The grid prints its actual peak RSS per cell, so this is now observed.
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
print(f"RAM: {ram:.1f} GB   (worst cell measured at ~5.0 GiB)")
if ram < 8:
    print("  [warn] tight. The run resumes per cell, so a crash costs only the cell in flight.")

## 2. Datasets\n\nPaths only -- needed because `cache_key` hashes them, so they must match what produced the caches. The Drive-cached CelebA zip is used first, so no kaggle.json.

In [ ]:
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT
assert CELEBA_OK, "CelebA unavailable -- the Drive-cached zip should make this credential-free"
print("datasets ready |", os.environ["WATERBIRDS_ROOT"], "|", CELEBA_ROOT)

## 3. Gate: validator + what is actually on Drive

In [ ]:
import glob

rc = subprocess.run([sys.executable, "-m", "study_robust_train.validate_grid"],
                    capture_output=True, text=True)
print(rc.stdout[-1500:])
assert rc.returncode == 0, "grid validator FAILED"

print()
for c in ("cache_clip", "cache_resnet", "cache_frozen"):
    n = len(glob.glob(f"results/{c}/*"))
    sz = sum(os.path.getsize(p) for p in glob.glob(f"results/{c}/*") if os.path.isfile(p)) / 1e9
    print(f"  {c:14s} {n:4d} file(s), {sz:5.2f} GB")

## 4. Cache probe

Only the two cheapest cells are built here, so a missing cache surfaces in seconds rather than
after an hour. Each build aborts past three minutes: on a CPU runtime that means features are being
*computed*, which is hours of silent work. The grid then loads the remaining cells one at a time.

In [ ]:
from study_robust_train.datasets import build_griddata

def cfg_for(dataset):
    base = {"clip":   {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cpu",
                       "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cpu", "epochs": 10, "lr": 1e-3, "batch_size": 128,
                       "cache_dir": "results/cache_resnet"},
            "frozen": {"device": "cpu", "cache_dir": "results/cache_frozen",
                       "batch_size": 128, "num_workers": 4}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                           "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
        base["resnet"]["max_train"] = CELEBA_RESNET_MAX_TRAIN
    return base

SLOW_SECONDS = 180

def build_cell(bb, ds):
    """One (backbone, dataset) GridData. A cache hit is seconds of Drive read; slower than that
    means this CPU runtime started COMPUTING features, which is hours of silent work."""
    t = time.time()
    gd = build_griddata(ds, bb, cfg_for(ds), seed=0)
    el = time.time() - t
    print(f"[loaded] {bb:14s}/{ds:10s} d={gd.train[0].shape[1]:5d} "
          f"train={gd.train[0].shape[0]:6d} eval={gd.eval_domain[0].shape[0]:6d} ({el:.0f}s)",
          flush=True)
    assert el < SLOW_SECONDS, (f"{bb}/{ds} took {el:.0f}s -- COMPUTED, not loaded. That cache is "
                               f"missing; re-extract it on a GPU runtime first.")
    return gd

import gc
for ds in DATASETS:                       # probe the cheapest backbone on both datasets
    gd = build_cell(BACKBONES[1], ds); del gd; gc.collect()
print()
print("cache verified on the probe cells; the grid loads the rest one at a time")

## 5. Why AFR collapses on CelebA (minutes -- run this first)

AFR scores 0.688-0.890 worst-group on Waterbirds and 0.013-0.029 on CelebA. That is not a method
losing, it is a degenerate head, and it is currently unexplained: **two hypotheses have been
tested and refuted** -- an in-sample stage-1 ERM (the corrected version scored *worse*), and
CelebA's class imbalance with an untuned gamma (AFR still improved, and gamma changed nothing).

So this measures rather than argues. The quantity to watch is the Kish effective sample size: on
synthetic data gamma=2.0 already collapses 1,333 weighted points to an ESS of 13. If CelebA's
9,957-row reweighting split collapses to a similar order against 2048 feature dimensions, the head
is underdetermined and the collapse is explained. `gamma=0` is the control -- it is unweighted AFR,
i.e. plain ERM on that split, so if worst-group accuracy is already near zero there then the
weighting is not the cause and the answer lies elsewhere.

This is cheap (cache hits, one head fit per gamma) and independent of the long run below.

In [ ]:
from study_robust_train.diagnose_afr import diagnose

afr_diag = {}
for bb in BACKBONES:
    afr_diag[(bb, "celeba")] = diagnose(build_cell(bb, "celeba"))
afr_diag[("resnet50_erm", "waterbirds")] = diagnose(build_cell("resnet50_erm", "waterbirds"))
print()
print("The Waterbirds cell is the control: AFR is healthy there, so anything that differs")
print("only on CelebA is a candidate cause.")

## 6. The ablation (~4 h at 3 seeds, streamed and resumable)

**Seeds.** Default 3, matching the published table's protocol so the new cells are directly
comparable to it. R1.4's request for more seeds is already met by the grid, which ran 5; and for
`erm` and `afr` the seed is inert anyway (lbfgs ignores `random_state`, so their across-seed SD is
exactly 0.000). Set `CAL_SEEDS = (0, 1, 2, 3, 4)` if you would rather have every table at five --
it costs roughly 60% more time and buys variance only for `dfr`, `groupdro_ll` and
`balanced_subsample`.

Written to a NEW file so the June ablation survives untouched as evidence. Re-run this cell to
continue after a disconnect; finished cells are read back and skipped.

In [ ]:
from study_robust_train.calibration_ablation import run_ablation_streaming

CAL_SEEDS = (0, 1, 2)          # see above; (0, 1, 2, 3, 4) for five
CAL_CSV   = "results/study/calibration_ablation_4bb.csv"
cal_keys  = [(bb, ds) for ds in DATASETS for bb in BACKBONES]   # Waterbirds first: cheap cells early

t = time.time()
cal = run_ablation_streaming(cal_keys, build_cell, methods=METHODS, scores=SCORES,
                             rho_sweep=RHO_SWEEP, seeds=CAL_SEEDS, n_splits=N_SPLITS,
                             alpha=ALPHA, cell_csv=CAL_CSV)
print(f"\nelapsed {(time.time() - t) / 60:.0f} min")
print(f"records  : {len(cal['records']):,}")
print(f"excluded : {len(cal['excluded'])} arm(s)")
print(f"failed   : {cal['failed']}")
by = {}
for r in cal["records"]:
    by[r["gate_status"]] = by.get(r["gate_status"], 0) + 1
print("gate_status:", by)

done = sorted({(r["backbone"], r["dataset"]) for r in cal["records"]})
print(f"\ncells complete: {len(done)}/{len(cal_keys)}")
if len(done) < len(cal_keys):
    print("Re-run THIS cell to continue.")
else:
    print(f"persisted -> {CAL_CSV}")

## 7. C1/C2/C3 -- with and without the gated arms

C1 is the paper's claim: under Mondrian every method reaches target, under `marginal_split` every
method falls significantly below. Both readings are printed. Where they disagree, the disagreement
is the finding -- report it, do not pick the friendlier one.

In [ ]:
def show(v, label):
    print()
    print("=" * 78)
    print(label)
    print("=" * 78)
    for key in sorted(v):
        c1 = v[key]["C1"]
        print(f"\n  {key[0]}/{key[1]}   C1_holds={c1['C1_holds']}")
        print(f"    {'method':20s} {'mondrian':>20s} {'marginal_split':>20s} {'shortfall':>10s}")
        for m, row in sorted(c1["methods"].items()):
            mo, ms = row["mondrian"], row["marginal_split"]
            print(f"    {m:20s} {mo['mean']:.3f} [{mo['ci'][0]:.3f},{mo['ci'][1]:.3f}]"
                  f"  {ms['mean']:.3f} [{ms['ci'][0]:.3f},{ms['ci'][1]:.3f}]"
                  f"  {row['split_shortfall']:+10.3f}")

show(cal["verdicts"], "MAIN -- gated arms excluded")
if "verdicts_with_excluded" in cal:
    show(cal["verdicts_with_excluded"], "SENSITIVITY (R2.4) -- gated arms included")
else:
    print("\nNo arm was gated out, so the two readings coincide.")

## 8. What is on Drive now

In [ ]:
sh(f"ls -la {DRIVE_CACHE}/study/", check=False)

## 9. Then

Download `calibration_ablation_4bb.csv` and the AFR diagnostic output. Remaining after this: the
reviewer statistics are already wired (`study_robust_train/reanalysis.py`) and only need pointing
at the new CSV; then the text -- R2.5, R2.6, R1.5, the title/abstract reframing from an absolute
dichotomy to dominance, the 20-page trim, and the response letter.